In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("Library berhasil di-import!")

Library berhasil di-import!


In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

#menampilkan 5 baris pertama dari data train
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
# [RESTART DATA GABUNGAN AGAR AMAN JIKA DI-RUN BERKALI-KALI]
test_ids = test_df['PassengerId']
full_df = pd.concat([train_df, test_df], sort=False).reset_index(drop=True)

# 1. Ekstraksi Title
full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Col': 'Rare', 'Major': 'Rare', 'Rev': 'Rare', 'Sir': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Lady': 'Rare',
    'Countess': 'Rare', 'Capt': 'Rare', 'Dr': 'Rare', 'Mlle': 'Miss',
    'Mme': 'Mrs', 'Ms': 'Miss'
}
full_df['Title'] = full_df['Title'].map(title_mapping)

# 2. Fitur Ukuran Keluarga
full_df['FamilySize'] = full_df['SibSp'] + full_df['Parch'] + 1
full_df['IsAlone'] = (full_df['FamilySize'] == 1).astype(int)

# 3. [UPGRADE] Frekuensi Tiket (Mendeteksi rombongan non-keluarga)
full_df['Ticket_Frequency'] = full_df.groupby('Ticket')['Ticket'].transform('count')

# 4. Ekstrak huruf depan Cabin sebagai Dek
full_df['Deck'] = full_df['Cabin'].str[0].fillna('U')

print("Cell 3: Feature Engineering Tingkat Lanjut Selesai!")

Feature Engineering selesai!


<>:6: SyntaxWarning: invalid escape sequence '\.'
<>:6: SyntaxWarning: invalid escape sequence '\.'
C:\Users\User\AppData\Local\Temp\ipykernel_28068\276377968.py:6: SyntaxWarning: invalid escape sequence '\.'
  full_df['Title'] = full_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


In [ ]:
# 1. Imputasi Nilai Hilang (Sama seperti sebelumnya)
full_df['Age'] = full_df.groupby(['Title', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
full_df['Fare'] = full_df.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
full_df['Embarked'] = full_df['Embarked'].fillna('S')

# 2. [UPGRADE] Pengelompokan Umur (Age Binning)
# Membagi umur menjadi 5 kelompok: Balita/Anak, Remaja, Dewasa Muda, Dewasa, Lansia
full_df['Age_Bin'] = pd.cut(full_df['Age'], bins=[0, 12, 20, 40, 60, 120], labels=[0, 1, 2, 3, 4]).astype(int)

# 3. [UPGRADE] Pengelompokan Tarif (Fare Binning)
# Membagi tarif berdasarkan kuartil (4 bagian sama rata)
full_df['Fare_Bin'] = pd.qcut(full_df['Fare'], 4, labels=[0, 1, 2, 3]).astype(int)

print("Cell 4: Data Cleaning & Binning Selesai!")

Age         0
Fare        0
Embarked    0
Title       0
dtype: int64

In [ ]:
# 1. One-Hot Encoding secara aman
categorical_cols = ['Sex', 'Embarked', 'Title', 'Deck']
cols_to_dummy = [col for col in categorical_cols if col in full_df.columns]

if cols_to_dummy:
    full_df = pd.get_dummies(full_df, columns=cols_to_dummy, drop_first=True)

# Ubah boolean jadi integer (1/0)
cols_to_convert = [c for c in full_df.columns if full_df[c].dtype == 'bool']
if cols_to_convert:
    full_df[cols_to_convert] = full_df[cols_to_convert].astype(int)

# 2. Pisahkan kembali train dan test
train_clean = full_df[full_df['Survived'].notnull()].copy()
test_clean = full_df[full_df['Survived'].isnull()].copy()

# 3. Hapus kolom mentah, TERMASUK Age dan Fare asli karena sudah diganti dengan Age_Bin dan Fare_Bin
drop_cols = ['PassengerId', 'Survived', 'Name', 'Ticket', 'Cabin', 'Age', 'Fare']

X = train_clean.drop(columns=drop_cols, errors='ignore')
y = train_clean['Survived'].astype(int)
X_test = test_clean.drop(columns=drop_cols, errors='ignore')

# 4. Samakan bentuk matriks
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

print(f"Cell 5: Berhasil! Jumlah Fitur yang akan dipelajari: {X.shape[1]}")

Cell 5 Berhasil! Jumlah Fitur Siap Pakai: 22


In [ ]:
from sklearn.model_selection import GridSearchCV

print("Sedang mencari parameter model terbaik (Hyperparameter Tuning)...")

# 1. Tentukan rentang parameter yang ingin diuji
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 5, 6, 8],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 2. Setup GridSearchCV
rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                           cv=5, scoring='accuracy', n_jobs=-1)

# 3. Latih untuk mencari yang terbaik
grid_search.fit(X, y)

# 4. Ambil model terbaik
best_model = grid_search.best_estimator_

print(f"Akurasi Cross-Validation Terbaik: {grid_search.best_score_ * 100:.2f}%")
print(f"Parameter Terbaik: {grid_search.best_params_}")

Akurasi per lipatan (fold) : [83.8  82.58 82.58 82.58 84.83]
Akurasi Rata-rata : 83.28%
Model berhasil dilatih dengan seluruh data train!


In [ ]:
# 1. Prediksi menggunakan model terbaik hasil GridSearch
predictions = best_model.predict(X_test)

# 2. Susun DataFrame (pastikan ejaan PassengerId benar)
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': predictions
})

# 3. Ekspor ke CSV
submission.to_csv('submission_titanic_v2.csv', index=False)

print("File 'submission_titanic_v2.csv' berhasil dibuat!")
submission.head()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2017467106.py, line 13)